# ClimaCity Paris -- Jour 1
## Exploration batch : l'API RDD et l'API DataFrame

**Module** : Traitement de donnees massives avec Apache Spark et PySpark  
**Duree** : 1 journee (6 heures effectives)  
**Prerequis** : Python intermediaire, notions de Pandas et NumPy

---

Ce notebook couvre l'integralite du Jour 1 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : l'API bas niveau RDD, la semantique de l'evaluation paresseuse, les transformations et actions fondamentales, et une premiere lecture du Spark UI.
- **Partie 2 -- Apres-midi (3 h)** : l'API haut niveau DataFrame, le nettoyage des donnees, la jointure avec les observations meteorologiques, la persistance en memoire et l'ecriture en Parquet partitionne.

> **Convention dans ce notebook**  
> Les cellules marquees `# [EXERCICE]` contiennent une consigne. Vous devez completer le code avant de passer a la cellule suivante.  
> Les cellules marquees `# [CORRECTION]` proposent une solution possible -- ne les regardez qu'apres avoir tente.


> ### Note d'adaptation du projet (à lire avant de commencer)
>
> L'énoncé suppose un extrait Velib' 2022-2023 « distribué aux étudiants ». Cet extrait
> n'est pas disponible publiquement. La seule source ouverte accessible est l'historique
> `lovasoa/historique-velib-opendata`, qui couvre **du 26 novembre 2020 au 9 avril 2021**
> (≈ 10,8 millions de relevés, donc un volume comparable aux « 12 millions » annoncés).
>
> Conséquences, appliquées de façon cohérente dans les 6 carnets :
>
> | Énoncé | Ce rendu |
> |--------|----------|
> | 2022 = entraînement, 2023 = test | découpage **temporel 85 % / 15 %** (prévu par le carnet 5 en cas de données absentes) |
> | données 2022 puis 2023 en Delta | novembre 2020 - février 2021, puis ajout de mars - avril 2021 |
> | météo SYNOP Météo-France | **Open-Meteo** (nouvelle source indiquée par l'enseignant le 26/06) |
> | `station_id` fourni | reconstitué : entier dense trié par nom (voir `scripts/velib_prep.py`) |
>
> Attention aussi à un biais d'analyse : cette période correspond à l'hiver 2020-2021,
> marqué par les restrictions sanitaires (couvre-feu, confinement en mars-avril 2021).
> Les usages observés ne sont pas ceux d'une année « normale ».

---
## Section 0 -- Configuration de l'environnement

Toutes les constantes du projet sont declarees ici. Adaptez les chemins si necessaire,
puis executez cette cellule avant toute autre.


In [1]:
import os
import sys
import time
from pathlib import Path

# ── Chemins de donnees ──────────────────────────────────────────────────────
DATA_DIR       = Path("../data")                          # racine des donnees
SCRIPTS_DIR    = Path("../scripts")                       # modules utilitaires (velib_prep.py)
VELIB_DIR      = DATA_DIR / "velib"
VELIB_RAW_DIR  = VELIB_DIR / "raw"                        # CSV bruts (RDD)
VELIB_PARQ_DIR = VELIB_DIR / "parquet"                    # Parquet partitionne
STATIONS_CSV   = VELIB_DIR / "stations_info.csv"
METEO_CSV      = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR     = DATA_DIR / "output"                     # ecriture des resultats

# ── Parametres Spark ────────────────────────────────────────────────────────
APP_NAME       = "ClimaCity-Paris"
MASTER         = "local[*]"                              # toutes les CPU locales
SHUFFLE_PARTS  = 8                                       # adapte aux volumes du cours
SEED           = 42

# Les chemins de donnees n'existent qu'apres la section 0.1 (collecte) :
# des "MANQUANT" a la toute premiere execution sont donc normaux.
for p in [VELIB_RAW_DIR, VELIB_PARQ_DIR, STATIONS_CSV, METEO_CSV]:
    status = "OK" if p.exists() else "MANQUANT"
    print(f"[{status}] {p}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] {OUTPUT_DIR} (cree si absent)")

[MANQUANT] ../data/velib/raw
[MANQUANT] ../data/velib/parquet
[MANQUANT] ../data/velib/stations_info.csv
[MANQUANT] ../data/meteo/paris_montsouris_horaire.csv
[OK] ../data/output (cree si absent)


---
## Section 0.1 -- Collecte des données Vélib'

Deux sources complémentaires sont nécessaires :

- **L'historique de disponibilité** (`lovasoa/historique-velib-opendata`, un relevé
  toutes les ~15 min par station) : archive `stations.zip` de la release GitHub.
- **Les informations de stations** : l'API GBFS de Vélib' Métropole (sans authentification)
  fournit le *code de station*, dont les premiers chiffres donnent l'arrondissement
  (`16107` -> 16e). L'historique, lui, ne contient ni identifiant ni arrondissement.

J'ai isolé le téléchargement et la remise en forme dans `scripts/velib_prep.py` (fonctions typées et
documentées), pour que ce carnet reste centré sur Spark. Ce script se limite à l'**acquisition** : tout
traitement analytique passe par Spark.

Les cellules ci-dessous sont **idempotentes** : si les fichiers existent déjà, rien n'est
retéléchargé (relancer le carnet est donc rapide).

In [2]:
# ── 1. Historique : téléchargement de l'archive ──────────────────────────────
# Le module velib_prep est dans ../scripts : on l'ajoute au chemin de recherche Python.
sys.path.insert(0, str(SCRIPTS_DIR))
import velib_prep

# telecharger() écrit d'abord un fichier .part puis le renomme : un téléchargement
# interrompu ne laisse jamais un fichier tronqué qui serait pris pour valide.
# (Pourquoi c'est important : 236 Mo, une coupure réseau est plausible.)
ZIP_HISTORIQUE = velib_prep.telecharger(
    velib_prep.HISTORIQUE_ZIP_URL, VELIB_DIR / "historique_stations.zip"
)
print(f"Archive : {ZIP_HISTORIQUE}  ({ZIP_HISTORIQUE.stat().st_size / 1_048_576:.0f} Mo)")

Archive : ../data/velib/historique_stations.zip  (225 Mo)


In [3]:
import pandas as pd

# ── 2. Référentiel des stations ─────────────────────────────────────────────
# J'utilise Pandas ici (contrainte n°4) : le référentiel compte ~1 400 lignes, bien sous les
# 10 000 autorisées, et cette étape ne fait qu'acquérir des données.
if STATIONS_CSV.exists():
    df_stations = pd.read_csv(STATIONS_CSV, sep=";")
    print("Référentiel déjà présent, relecture.")
else:
    # L'API GBFS donne le code de station actuel ; construire_referentiel() l'associe à
    # l'historique par nom, puis par proximité géographique pour les stations renommées.
    gbfs = velib_prep.charger_stations_gbfs()
    print(f"  {len(gbfs)} stations dans l'API GBFS actuelle")
    df_stations = velib_prep.construire_referentiel(ZIP_HISTORIQUE, gbfs)
    df_stations.to_csv(STATIONS_CSV, index=False, sep=";")
    print(f"  Sauvegardé : {STATIONS_CSV}")

print(f"  {len(df_stations)} stations dans l'historique")
print(f"  Capacité totale du réseau : {df_stations['capacity'].sum():,} bornettes")
df_stations.head(5)

  1518 stations dans l'API GBFS actuelle


  Sauvegardé : ../data/velib/stations_info.csv
  1402 stations dans l'historique
  Capacité totale du réseau : 44,089 bornettes


,station_id,name,lat,lon,capacity,stationCode,code_arr,cle
0,1001,11 Novembre 1918 - 8 Mai 1945,48.80890,2.53824,36,45003,45,11 Novembre 1918 - 8 Mai 1945
1,1002,18 juin 1940 - Buzenval,48.86881,2.18543,25,25005,25,18 juin 1940 - Buzenval
2,1003,8 Mai 1945 - 10 Juillet 1940,48.78457,2.39790,30,44010,44,8 Mai 1945 - 10 Juillet 1940
3,1004,Abbeville - Faubourg Poissonnière,48.87922,2.34915,14,9002,9,Abbeville - Faubourg Poissonnière
4,1005,Abbé Carton - Plantes,48.82767,2.32092,25,14110,14,Abbé Carton - Plantes


In [4]:
# ── 3. Conversion au format CSV brut que lit la suite du carnet ──────────────
# Je produis un fichier .csv.gz par quinzaine, au format documenté en 1.2 :
#   station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
fichiers = sorted(VELIB_RAW_DIR.glob("*.csv.gz"))
if not fichiers:
    fichiers = velib_prep.convertir_historique(ZIP_HISTORIQUE, df_stations, VELIB_RAW_DIR)

# ── 4. Météo horaire (Open-Meteo, sans clé) ──────────────────────────────────
# La période météo est déduite des noms de fichiers (velib_AAAA-MM-A|B.csv.gz).
mois_debut = fichiers[0].name.split("_")[1][:7]
mois_fin   = fichiers[-1].name.split("_")[1][:7]
if not METEO_CSV.exists():
    fin_periode = pd.Period(mois_fin).end_time.strftime("%Y-%m-%d")
    velib_prep.telecharger_meteo(f"{mois_debut}-01", fin_periode, METEO_CSV)

taille_totale = sum(f.stat().st_size for f in fichiers) / 1_048_576
print(f"{len(fichiers)} fichiers Vélib' dans {VELIB_RAW_DIR}  ({taille_totale:.0f} Mo compressés)")
print(f"Période couverte : {mois_debut} -> {mois_fin}")
print(f"Météo : {METEO_CSV}")

10 fichiers Vélib' dans ../data/velib/raw  (195 Mo compressés)
Période couverte : 2020-11 -> 2021-04
Météo : ../data/meteo/paris_montsouris_horaire.csv


# **Nota bene**
- Le téléchargement (≈ 236 Mo) et la conversion (≈ 1 à 2 minutes) ne sont faits qu'à la première exécution.
- Le fichier source est une archive unique, alors que l'énoncé parle d'un `.csv.gz` par quinzaine.
  La conversion recrée les 10 fichiers attendus. Cela joue sur le parallélisme des RDD : un fichier `.gz`
  ne se découpe pas, donc chaque fichier forme **une partition**.
- J'écarte à cette étape les relevés de stations hors service (`operative = False`, ≈ 1,7 %) : un « zéro
  vélo » sur une station éteinte ne signale aucune rupture.

---
# PARTIE 1 -- L'API RDD (matin)

## 1.1 La SparkSession et le SparkContext

### Pourquoi Spark ?

Pandas est un outil remarquable pour les donnees qui tiennent en memoire sur une seule
machine. Mais il atteint ses limites lorsque le volume depasse quelques gigaoctets, ou
lorsque le traitement doit s'executer en parallele sur plusieurs coeurs ou plusieurs noeuds.

Apache Spark repond a ce besoin en proposant un modele de calcul distribue et tolerant aux
pannes, dans lequel les donnees sont representees comme des collections immuables et
partitionnees, les **RDD** (*Resilient Distributed Datasets*).

Meme en mode local -- sur une seule machine, comme c'est le cas dans ce cours -- Spark
parallelise les traitements sur tous les coeurs disponibles et permet de travailler sur
des volumes qui depassent la memoire vive grace a la gestion automatique du debordement
sur disque.

### Le point d'entree : `SparkSession`

Depuis Spark 2.0, `SparkSession` est le point d'entree unifie. Il encapsule le
`SparkContext` (execution RDD), le `SQLContext` (SQL et DataFrame) et le
`HiveContext`. On n'en cree qu'une par application.


In [5]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

"""
Exercice 1 :
------------
Initialiser une session Spark en assignant :
- un nombre de partitions
- une taille de mémoire
- en supprimant la trace de la console
"""
spark = (
    SparkSession.builder
    .appName(APP_NAME)                                # nom visible dans le Spark UI
    .master(MASTER)                                   # local[*] : un thread par coeur
    # Nombre de partitions APRES un shuffle (defaut = 200). Pour ~10 M de lignes sur une
    # machine locale, 200 partitions créeraient des centaines de micro-tâches dont
    # l'ordonnancement coûterait plus cher que le calcul : on descend à 8.
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    # Mémoire du driver. En mode local le driver EST l'exécuteur : c'est cette valeur
    # qui limite le cache et les shuffles (le conteneur Docker en alloue 9 Go).
    .config("spark.driver.memory", "6g")
    # Les horodatages du projet sont en UTC (schéma cible) : on fixe le fuseau de la session
    # pour qu'un même instant soit lu et affiché identiquement partout (jointure météo, SQL...).
    .config("spark.sql.session.timeZone", "UTC")
    # Supprime les barres de progression [Stage x:>...] dans la sortie du notebook.
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()                                    # réutilise la session si elle existe
)

# Le SparkContext est accessible depuis la session
sc = spark.sparkContext
sc.setLogLevel("WARN")    # reduit les logs INFO

print(f"Spark version    : {spark.version}")
print(f"Python version   : {sc.pythonVer}")
print(f"Master           : {sc.master}")
print(f"Coeurs detectes  : {sc._jsc.sc().defaultParallelism()}")
print(f"\nSpark UI disponible sur : http://localhost:4040")

Spark version    : 3.5.3
Python version   : 3.11
Master           : local[*]
Coeurs detectes  : 24

Spark UI disponible sur : http://localhost:4040


> **Spark UI** : ouvrez l'adresse affichee ci-dessus dans votre navigateur. Vous trouverez
> l'onglet **Jobs** (vide pour l'instant), **Stages**, **Storage** et **Environment**.
> Gardez-le ouvert en permanence pendant cette journee.

### Structure d'un cluster Spark (meme en local)

```
SparkSession (driver)
  |
  +-- SparkContext
        |
        +-- Executor 1 (Thread pool)
        |     +-- Task 1-1
        |     +-- Task 1-2
        +-- Executor 2 (Thread pool)
              +-- Task 2-1
              +-- Task 2-2
```

En mode `local[*]`, le driver et les executeurs partagent le meme processus JVM.
Chaque coeur logique correspond a un *slot* d'execution de taches.


---
## 1.2 Chargement des donnees brutes avec `sc.textFile()`

Le jeu de donnees brut se presente sous forme de fichiers CSV compresses (`.csv.gz`),
un par quinzaine de jours environ. Chaque ligne decrit l'etat d'une station a un instant
donne. Voici le format :

```
station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
1001;Bir-Hakeim - Grenelle;75015;25;12;3;10;2022-01-01T00:14:52+00:00
1002;Javel - Andre Citroen;75015;30;0;0;30;2022-01-01T00:14:55+00:00
...
```

L'API `sc.textFile()` retourne un **RDD de chaines de caracteres** -- une ligne du fichier
par element. Aucune interpretation du contenu n'est effectuee a ce stade.


In [6]:
"""
Exercice 2 :
------------
Charger les ficbiers de log des Velib
sc.textFile() accepte un chemin vers un fichier, un repertoire, ou un glob
Il decompresse automatiquement les .gz
"""
# Le glob "*.csv.gz" désigne les 10 fichiers d'un coup. textFile() est PARESSEUX : il
# ne lit rien, il construit seulement un RDD[str] (une chaîne = une ligne de fichier).
raw_rdd = sc.textFile(str(VELIB_RAW_DIR / "*.csv.gz"))

"""
Exercice 3 :
------------
Compter le nombre de lignes du RDD.
count() est une ACTION -- elle declenche le calcul
"""
# Première action : c'est ici que Spark lit et décompresse réellement les fichiers.
# Le résultat inclut les 10 lignes d'en-tête (une par fichier) : on les retirera en 1.3.
n_lignes_brutes = raw_rdd.count()
print(f"Lignes brutes (en-têtes inclus) : {n_lignes_brutes:,}")

"""
Exercice 4 :
------------
Apercu des 5 premieres lignes -- take() est aussi une action
"""
# take(n) ne lit que le nombre de partitions nécessaire pour obtenir n lignes : bien
# moins coûteux que collect(), à réserver aux petits RDD.
for ligne in raw_rdd.take(5):
    print(ligne)

Lignes brutes (en-têtes inclus) : 10,769,274


station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
1098;Benjamin Godard - Victor Hugo;16;35;4;5;26;2020-11-26T12:59:00+00:00
1031;André Mazet - Saint-André des Arts;6;55;23;4;28;2020-11-26T12:59:00+00:00
1225;Charonne - Robert et Sonia Delauney;11;20;0;0;20;2020-11-26T12:59:00+00:00
2330;Toudouze - Clauzel;9;21;0;1;20;2020-11-26T12:59:00+00:00


### Observations

- `sc.textFile()` est **instantane** : il ne lit pas encore les fichiers. Il cree
  seulement un *plan de calcul* (un RDD logique).
- `count()` declenche la lecture et le comptage -- c'est la premiere **action**.
- Les fichiers sont decoupes en **partitions** (une par bloc HDFS ou par fichier local).
  Le nombre de partitions determine le parallelisme maximal.

Verifions le nombre de partitions :


In [7]:
print(f"Nombre de partitions : {raw_rdd.getNumPartitions()}")

# En mode local, Spark cree typiquement une partition par coeur,
# ou une par fichier si le nombre de fichiers est superieur au nombre de coeurs.
# On peut forcer un repartitionnement :
raw_rdd_8p = raw_rdd.repartition(8)
print(f"Apres repartition(8) : {raw_rdd_8p.getNumPartitions()} partitions")
# Note : repartition() est une transformation -- elle ne s'execute pas encore.


Nombre de partitions : 10
Apres repartition(8) : 8 partitions


---
## 1.3 Transformations elementaires : `map`, `filter`

Un RDD est une collection **immuable** et **paresseuse** : les transformations ne
s'executent qu'au moment ou une action les requiert.

### Separation de l'en-tete

La premiere ligne de chaque fichier est un en-tete. Il faut l'eliminer avant de parser
les donnees.


In [8]:
"""
Exercice 5 :
------------
Recuperer l'en-tete pour connaitre les colonnes
Il existe une fonction spéciale pour cela
"""
# first() est une action qui renvoie le premier élément du RDD : ici l'en-tête du
# premier fichier. Les 10 fichiers ayant le même en-tête, une seule valeur suffit
# pour les reconnaître toutes.
entete = raw_rdd.first()
print("En-tête :", entete)

"""
Exercice 6 :
------------
Filtrer les lignes d'en-tete (toutes les lignes identiques a la premiere)
filter() retourne un nouveau RDD contenant uniquement les elements
pour lesquels la fonction retourne True

1) Qu'est-ce qui s'affiche à la sortie de `filter` ?
2) Comment affijhcer le nombre de lignes de donnees ?
"""
data_rdd = raw_rdd.filter(lambda ligne: ligne != entete)

# Réponse 1 : `filter` renvoie un NOUVEAU RDD (objet PipelinedRDD), pas des données :
#   une transformation ne calcule rien, elle enrichit le plan (le DAG).
print("Sortie de filter :", data_rdd)

# Réponse 2 : il faut une ACTION, ici count(), pour déclencher le calcul.
n_donnees = data_rdd.count()
print(f"Lignes de données : {n_donnees:,}  (= {n_lignes_brutes:,} - {n_lignes_brutes - n_donnees} en-têtes)")

En-tête : station_id;nom_station;code_arrondissement;capacite;velos_meca;velos_elec;bornettes_libres;horodatage
Sortie de filter : PythonRDD[10] at RDD at PythonRDD.scala:53


Lignes de données : 10,769,264  (= 10,769,274 - 10 en-têtes)


### Parsing des lignes avec `map()`

`map(f)` applique la fonction `f` a chaque element du RDD et retourne un nouveau RDD
de meme longueur contenant les resultats. C'est une transformation *un-pour-un*.


In [9]:
from datetime import datetime, timezone

COLONNES = [
    "station_id", "nom_station", "code_arr", "capacite",
    "velos_meca", "velos_elec", "bornettes_libres", "horodatage"
]

"""
Exercice 7  :
------------
"""
def parse_ligne(line: str) -> dict | None:
    """Parse une ligne CSV du fichier Velib' brut.

    Args:
        line: Chaine brute separee par des points-virgules.

    Returns:
        Dictionnaire avec les champs types, ou None si la ligne est malformee.

    Example:
        >>> parse_ligne("1100;Godard;16;35;1;0;34;2021-01-01T00:47:00+00:00")["capacite"]
        35
        >>> parse_ligne("ligne;cassee") is None
        True
    """
    champs = line.split(";")
    # Une ligne saine a exactement 8 champs. Tout autre nombre trahit une ligne tronquée
    # ou un nom de station contenant ";" : on la rejette plutôt que de deviner.
    if len(champs) != len(COLONNES):
        return None
    try:
        return {
            "station_id":       int(champs[0]),
            "nom_station":      champs[1],
            "code_arr":         int(champs[2]),
            "capacite":         int(champs[3]),
            "velos_meca":       int(champs[4]),
            "velos_elec":       int(champs[5]),
            "bornettes_libres": int(champs[6]),
            "horodatage":       champs[7],   # gardé en texte ; typé plus tard en DataFrame
        }
    except ValueError:
        # int() échoue sur une valeur non numérique : ligne malformée -> None
        return None


# Appliquer le parsingau jeu de données
# map() est "un pour un" : chaque ligne devient un dict, ou None si elle est malformée.
parsed_rdd = data_rdd.map(parse_ligne)

# Supprimer les lignes malformees (None)
# On filtre après le map pour ne pas parser deux fois la même ligne.
valid_rdd = parsed_rdd.filter(lambda d: d is not None)

# Compter les lignes
n_valides = valid_rdd.count()
print(f"Lignes valides    : {n_valides:,}")
print(f"Lignes rejetées   : {n_donnees - n_valides:,}")

# Afficher les deux premières
for d in valid_rdd.take(2):
    print(d)

Lignes valides    : 10,769,264
Lignes rejetées   : 0


{'station_id': 1098, 'nom_station': 'Benjamin Godard - Victor Hugo', 'code_arr': 16, 'capacite': 35, 'velos_meca': 4, 'velos_elec': 5, 'bornettes_libres': 26, 'horodatage': '2020-11-26T12:59:00+00:00'}
{'station_id': 1031, 'nom_station': 'André Mazet - Saint-André des Arts', 'code_arr': 6, 'capacite': 55, 'velos_meca': 23, 'velos_elec': 4, 'bornettes_libres': 28, 'horodatage': '2020-11-26T12:59:00+00:00'}


> **Examen du Spark UI** : allez dans l'onglet **Jobs**. Vous devriez voir deux jobs
> termines. Cliquez sur le dernier (le `count()`). Dans la vue **Stages**, observez :
>
> - Le nombre de taches (*tasks*) -- une par partition.
> - Le temps passe dans chaque tache.
> - La colonne **Input** : la quantite de donnees lues par chaque tache.
>
> Notez que les deux `count()` que nous avons declenches ont relu les fichiers
> depuis le disque a chaque fois. C'est le comportement par defaut -- nous verrons
> comment l'eviter avec `.cache()`.


---
## 1.4 Evaluation paresseuse : transformations vs actions

C'est le concept le plus important de Spark. Toutes les **transformations** sont
**paresseuses** : elles construisent un plan de calcul (un DAG), mais ne font rien.
Seules les **actions** declenchent l'execution.

| Transformations (paresseuses) | Actions (declenchent le calcul) |
|-------------------------------|----------------------------------|
| `map()`, `filter()`, `flatMap()` | `count()`, `collect()`, `take()` |
| `groupByKey()`, `reduceByKey()` | `first()`, `top()`, `reduce()` |
| `join()`, `union()`, `distinct()` | `saveAsTextFile()`, `foreach()` |
| `repartition()`, `coalesce()` | `countByValue()`, `countByKey()` |

Illustrons ce comportement :


In [10]:
import time

# --- Construction du plan (instantane) ---
t0 = time.perf_counter()

"""
Exercice 8 :
-----------
On veut effectuer un calcul en trois étapes :
1) Relever le nombre de stations non vides
2) Calculer le nombre de vélos et le taux d'occupation
3) Ne conserver les lignes avec taux d'occupation de moins de 10%
"""
# Étape 1 : on ne garde que les stations de capacité > 0 (les autres n'ont pas de
# taux d'occupation défini : division par zéro).
step1 = valid_rdd.filter(lambda d: d["capacite"] > 0)

# Étape 2 : on enrichit chaque relevé avec le nombre de vélos et le taux d'occupation,
# défini comme dans le schéma cible du projet : (capacité - bornettes libres) / capacité.
# On crée un NOUVEAU dict ({**d, ...}) : les RDD sont immuables, on ne modifie jamais
# un élément en place.
def ajouter_taux(d: dict) -> dict:
    """Ajoute nb_velos et taux_occupation à un relevé.

    Args:
        d: Relevé parsé (dict issu de parse_ligne), de capacité > 0.

    Returns:
        Copie du relevé avec les clés ``nb_velos`` et ``taux_occupation``.
    """
    return {
        **d,
        "nb_velos": d["velos_meca"] + d["velos_elec"],
        "taux_occupation": (d["capacite"] - d["bornettes_libres"]) / d["capacite"],
    }

step2 = step1.map(ajouter_taux)

# Étape 3 : les stations presque vides (< 10 %), celles qui intéressent l'équipe métier.
step3 = step2.filter(lambda d: d["taux_occupation"] < 0.10)

# Combien de temps a pris cette opération ?
t1 = time.perf_counter()
print(f"Construction du plan (3 transformations) : {(t1-t0)*1000:.1f} ms")

# --- Declenchement par une action ---
t2 = time.perf_counter()
"""
Exercice 9 :
------------
Compter le nombre de stations
"""
n = step3.count()
t3 = time.perf_counter()

# Combien de temps a pris cette opération
print(f"Execution de count() (lecture + calcul) : {(t3-t2):.2f} s")
print(f"Lignes avec taux_occupation < 10 % : {n:,}")
# Observation : Spark construit le plan en quelques millisecondes sans rien calculer, puis count()
# prend des dizaines de secondes. C'est l'évaluation paresseuse.

Construction du plan (3 transformations) : 0.1 ms


Execution de count() (lecture + calcul) : 3.58 s
Lignes avec taux_occupation < 10 % : 1,722,926


### Le DAG (Directed Acyclic Graph)

Spark traduit votre chaine de transformations en un graphe oriente acyclique.
Ce graphe est optimise avant l'execution par le **Catalyst optimizer** (pour les
DataFrames) ou execute tel quel (pour les RDD).

```
sc.textFile()  -->  filter(header)  -->  map(parse)  -->  filter(None)
    -->  filter(capacite > 0)  -->  map(taux)  -->  filter(valide)
    -->  count()  [ACTION]
```

> **Spark UI -> Jobs -> dernier job -> DAG Visualization** : vous voyez exactement
> ce graphe. Les boites bleues sont les **stages** (separees par des shuffles).
> Une fleche grise entre deux stages indique un shuffle (transfert de donnees
> entre executeurs).


In [11]:
# On va reutiliser step3 plusieurs fois -> bonne occasion de le mettre en cache
# .cache() est equivalent a .persist(StorageLevel.MEMORY_ONLY)
step3.cache()

# Premier acces : lecture disque + mise en cache
t0 = time.perf_counter()
n1 = step3.count()
t1 = time.perf_counter()
print(f"Premier count() (lecture disque) : {t1-t0:.2f} s -- {n1:,} lignes")

# Deuxieme acces : depuis le cache en memoire
t2 = time.perf_counter()
n2 = step3.count()
t3 = time.perf_counter()
print(f"Deuxieme count() (depuis cache)  : {t3-t2:.2f} s -- {n2:,} lignes")

print(f"\nGain du cache : x{((t1-t0)/(t3-t2)):.1f}")
print("\nAllez dans Spark UI -> Storage pour voir le RDD en cache.")


Premier count() (lecture disque) : 4.20 s -- 1,722,926 lignes


Deuxieme count() (depuis cache)  : 0.21 s -- 1,722,926 lignes

Gain du cache : x19.5

Allez dans Spark UI -> Storage pour voir le RDD en cache.


---
## 1.5 Agregations avec paires cle-valeur : `reduceByKey`

L'une des operations les plus importantes de l'API RDD est `reduceByKey()`. Elle
regroupe les elements qui partagent la meme cle et combine leurs valeurs avec une
fonction associative.

### Objectif : top 10 des stations par nombre de snapshots

Nous voulons savoir quelles stations apparaissent le plus souvent dans notre historique
(indicateur indirect d'une bonne couverture de collecte).


In [12]:
"""
Exercice 10 :
------------
"""
# Etape 1 : transformer chaque enregistrement en paire (cle, valeur)
# Ici : (station_id, 1) -- on compte une occurrence par ligne
# Un RDD de paires (clé, valeur) est ce qui débloque reduceByKey, join, sortByKey...
paires_rdd = valid_rdd.map(lambda d: (d["station_id"], 1))

# Apercu
print("Exemples de paires :")
for p in paires_rdd.take(5):
    print(" ", p)

Exemples de paires :


  (1098, 1)
  (1031, 1)
  (1225, 1)
  (2330, 1)
  (1747, 1)


In [13]:
# Etape 2 : sommer les occurrences par station
# reduceByKey(f) applique f a toutes les valeurs qui ont la meme cle,
# en garantissant que f est appliquee localement avant le shuffle (combinaison locale)
# f doit être associative et commutative (a+b l'est) : c'est ce qui autorise Spark à
# combiner partiellement dans chaque partition avant de transférer quoi que ce soit.
comptage_rdd = paires_rdd.reduceByKey(lambda a, b: a + b)

# Etape 3 : trier par nombre decroissant et prendre le top 10
# La clé de tri (-compte, id) départage les ex æquo par identifiant : le résultat est
# alors déterministe d'une exécution à l'autre (utile pour comparer avec Pandas).
top10_rdd = comptage_rdd.sortBy(lambda kv: (-kv[1], kv[0]))

# Étape 4 : Prendre les dix preùières stations
# take(10) est l'action : elle déclenche tout le calcul (lecture, parsing, shuffle, tri).
top10 = top10_rdd.take(10)

print("Top 10 des stations par nombre de snapshots :")
print(f"{'Station ID':<12} {'Snapshots':>12}")
print("-" * 26)
for station_id, count in top10:
    print(f"{station_id:<12} {count:>12,}")

Top 10 des stations par nombre de snapshots :
Station ID      Snapshots
--------------------------
1001                7,866
1002                7,866
1004                7,866
1005                7,866
1006                7,866
1007                7,866
1008                7,866
1009                7,866
1010                7,866
1011                7,866


### `reduceByKey` vs `groupByKey` : une difference critique

`groupByKey()` regroupe *toutes* les valeurs en memoire avant de les reduire.
Sur de gros volumes, cela peut provoquer des `OutOfMemoryError`.
`reduceByKey()` effectue une **combinaison locale** avant le shuffle, ce qui reduit
drastiquement le volume de donnees transferees.

```
groupByKey()   : SHUFFLE toutes les valeurs -> reduire
reduceByKey()  : reduire localement -> SHUFFLE resultats partiels -> reduire
```

Regle pratique : **ne jamais utiliser `groupByKey()` quand `reduceByKey()` suffit**.


In [14]:
# Illustration de la difference de performance
# (Sur les petits volumes du cours, l'ecart peut etre faible -- il explose a l'echelle)

"""
Exercice 12 :
------------
Comparaison du temps d'execution de groupByKey() et reduceByKey() pour compter les occurrences par clef
"""
# Variante 1 : groupByKey rapatrie TOUTES les valeurs d'une clé (ici des milliers de « 1 »
# par station) avant de les sommer avec mapValues(sum). Pas de combinaison locale.
t0 = time.perf_counter()
paires_rdd.groupByKey().mapValues(sum).count()
t_group = time.perf_counter() - t0

# Variante 2 : reduceByKey somme d'abord dans chaque partition, puis ne transfère
# qu'UNE valeur partielle par clé et par partition : bien moins de données au shuffle.
t0 = time.perf_counter()
paires_rdd.reduceByKey(lambda a, b: a + b).count()
t_reduce = time.perf_counter() - t0

print(f"groupByKey + mapValues(sum) : {t_group:.2f} s")
print(f"reduceByKey                 : {t_reduce:.2f} s")
print(f"Rapport                     : x{t_group/t_reduce:.1f}")
# Lecture dans le Spark UI (onglet Stages) : comparez la colonne « Shuffle Write » des deux jobs.

groupByKey + mapValues(sum) : 3.58 s
reduceByKey                 : 3.44 s
Rapport                     : x1.0


---
## 1.6 `flatMap` et `join` entre deux RDD

### `flatMap` : une entree, zero ou plusieurs sorties

Contrairement a `map` (toujours une sortie par entree), `flatMap` peut retourner
un nombre quelconque d'elements pour chaque element d'entree. C'est utile pour
"eclater" des structures imbriquees.


In [15]:
# Exemple pedagogique : a partir de chaque snapshot, on veut produire
# autant de paires (heure_de_la_journee, taux_occupation) que necessaire
# pour alimenter une distribution horaire.
def extraire_heure_et_taux(record: dict) -> list[tuple]:
    """Extrait l'heure ISO et le taux d'occupation.

    Args:
        record: Dictionnaire d'un snapshot Velib'.

    Returns:
        Liste de paires (heure_int, taux_occupation) ou liste vide si erreur.

    Example:
        >>> extraire_heure_et_taux({"horodatage": "2021-01-01T08:47:00+00:00", "taux_occupation": 0.5})
        [(8, 0.5)]
        >>> extraire_heure_et_taux({"horodatage": "n/a"})
        []
    """
    try:
        # Format ISO "AAAA-MM-JJTHH:MM:SS+00:00" : les positions 11-12 sont l'heure.
        # Un découpage de chaîne est bien plus rapide que datetime.fromisoformat sur 10 M de lignes.
        return [(int(record["horodatage"][11:13]), record["taux_occupation"])]
    except (KeyError, ValueError, TypeError):
        # flatMap accepte une liste vide : la ligne disparaît silencieusement du résultat,
        # ce que map() ne saurait pas faire (il produirait un None à filtrer ensuite).
        return []

# Question : Quelle est la forme du résultat `heure_taux_rdd`
# Réponse : un RDD « plat » de tuples (heure, taux). flatMap concatène les listes renvoyées,
# donc il donne un tuple par relevé valide et aucune liste imbriquée.
# NB : je pars de `step2`, qui contient tous les relevés avec leur taux. `step3` ne garde que
# les stations presque vides (< 10 %) et fausserait le profil horaire.
heure_taux_rdd = step2.flatMap(extraire_heure_et_taux)

"""
Exercice 13 :
-------------
# Calculer la moyenne du taux d'occupation par heure de la journee
# Astuce : on somme (taux, 1) puis on divise
"""
# Une moyenne n'est pas associative, alors que le couple (somme, effectif) l'est : je l'agrège
# avec reduceByKey, puis je divise à la fin.
somme_count_rdd = heure_taux_rdd.map(lambda ht: (ht[0], (ht[1], 1)))
agregat_rdd     = somme_count_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
moyenne_rdd     = agregat_rdd.mapValues(lambda s: s[0] / s[1])
profil_horaire  = moyenne_rdd.sortByKey().collect()   # 24 éléments : collect() est sans risque

print("Taux d'occupation moyen par heure de la journee (toutes stations)")
print(f"{'Heure':<8} {'Taux moyen':>12}")
print("-" * 22)
for heure, taux in profil_horaire:
    barre = "#" * int(taux * 30)
    print(f"  {heure:02d}h    {taux:.4f}  {barre}")

Taux d'occupation moyen par heure de la journee (toutes stations)
Heure      Taux moyen
----------------------
  00h    0.3929  ###########
  01h    0.3943  ###########
  02h    0.3965  ###########
  03h    0.3966  ###########
  04h    0.3958  ###########
  05h    0.3961  ###########
  06h    0.3834  ###########
  07h    0.3658  ##########
  08h    0.3658  ##########
  09h    0.3667  ##########
  10h    0.3575  ##########
  11h    0.3439  ##########
  12h    0.3431  ##########
  13h    0.3451  ##########
  14h    0.3449  ##########
  15h    0.3395  ##########
  16h    0.3320  #########
  17h    0.3418  ##########
  18h    0.3449  ##########
  19h    0.3558  ##########
  20h    0.3720  ###########
  21h    0.3868  ###########
  22h    0.3911  ###########
  23h    0.3929  ###########


### `join` entre deux RDD

On peut joindre deux RDD de paires `(cle, valeur)` comme on joinderait deux tables.


In [16]:
"""
Exercice 14 :
-------------
"""
# RDD des stations avec leur nom : (station_id, nom_station)
# distinct() élimine les doublons : sans lui, chaque station apparaîtrait autant de fois
# que de relevés et la jointure multiplierait les lignes.
noms_rdd = valid_rdd.map(lambda d: (d["station_id"], d["nom_station"])).distinct()

# RDD du total de velos par station : (station_id, total_velos_disponibles)
velos_rdd = step2.map(lambda d: (d["station_id"], d["nb_velos"])).reduceByKey(lambda a, b: a + b)

# Join interne
# join() ne s'applique qu'à des RDD de paires et joint sur la clé : c'est un shuffle des deux côtés.
joint_rdd = noms_rdd.join(velos_rdd)
# Resultat : (station_id, (nom_station, total_velos))

# Top 5 des stations par volume de velos disponibles (cumule)
# takeOrdered évite de trier tout le RDD : chaque partition ne remonte que ses 5 meilleurs.
top5 = joint_rdd.takeOrdered(5, key=lambda kv: -kv[1][1])

print("Top 5 des stations par volume cumule de velos disponibles :")
for sid, (nom, total) in top5:
    print(f"  [{sid:4d}] {nom:<40} {total:>10,}")

Top 5 des stations par volume cumule de velos disponibles :
  [1351] Emeriau - Beaugrenelle                      422,058
  [1511] Grenelle - Dr Finlay                        398,057
  [1868] Parc André Citroën                          362,519
  [1138] Bourdonnais - Tour Eiffel                   352,836
  [2149] Regnault - Patay                            340,502


---
## 1.7 Comparaison Pandas vs Spark

Reprenons le calcul "top 10 des stations par nombre de snapshots" en Pandas et
comparons les temps et la demarche.


In [17]:
import pandas as pd
import glob

# --- Version Pandas ---
t0 = time.perf_counter()

fichiers = glob.glob(str(VELIB_RAW_DIR / "*.csv.gz"))
df_pandas = pd.concat(
    [pd.read_csv(f, sep=";", compression="gzip") for f in fichiers],
    ignore_index=True
)
top10_pandas = (
    df_pandas["station_id"].value_counts().head(10)
)
t_pandas = time.perf_counter() - t0

# --- Version Spark (deja calculee) ---
t0 = time.perf_counter()
top10_spark = top10_rdd.take(10)
t_spark = time.perf_counter() - t0   # depuis le cache

print(f"Pandas  : {t_pandas:.2f} s  --  {len(df_pandas):,} lignes chargees")
print(f"Spark   : {t_spark:.2f} s  --  depuis le cache RDD")
print()
print("Comparaison des resultats (top 5) :")
print(f"{'Station':>10}  {'Pandas':>12}  {'Spark':>12}")
for i in range(5):
    p_id, p_cnt = top10_pandas.index[i], top10_pandas.iloc[i]
    s_id, s_cnt = top10_spark[i]
    print(f"{p_id:>10}  {p_cnt:>12,}  {s_cnt:>12,}")


Pandas  : 5.92 s  --  10,769,264 lignes chargees
Spark   : 0.04 s  --  depuis le cache RDD

Comparaison des resultats (top 5) :
   Station        Pandas         Spark
      1098         7,866         7,866
      1716         7,866         7,866
      2391         7,866         7,866
      1912         7,866         7,866
      2350         7,866         7,866


In [18]:
# --- Analyse de la memoire ---
mem_pandas_mb = df_pandas.memory_usage(deep=True).sum() / 1_048_576
print(f"\nMemoire occupee par le DataFrame Pandas  : {mem_pandas_mb:.1f} MB")
print(f"Memoire disponible pour le reste de l'OS : limitee")
print()
print("Conclusion :")
print("  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)")
print("  - Spark devient avantageux quand :")
print("    * Le volume depasse la RAM disponible")
print("    * Le calcul peut etre parallelise sur un cluster")
print("    * On combine batch + streaming + ML dans le meme pipeline")
print("    * On travaille en equipe sur un cluster partage")

# Je libère le DataFrame Pandas (plusieurs Go) avant la suite : la JVM Spark et Python se
# partagent la RAM du conteneur.
import gc
del df_pandas
gc.collect()


Memoire occupee par le DataFrame Pandas  : 2246.9 MB
Memoire disponible pour le reste de l'OS : limitee

Conclusion :
  - Sur ce volume, Pandas est souvent plus rapide (pas d'overhead JVM/reseau)
  - Spark devient avantageux quand :
    * Le volume depasse la RAM disponible
    * Le calcul peut etre parallelise sur un cluster
    * On combine batch + streaming + ML dans le meme pipeline
    * On travaille en equipe sur un cluster partage


92